<a href="https://colab.research.google.com/github/asomers205/DS2002FA26/blob/main/notebooks/04-json-apis/2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _line item — one (item, qty) entry from an order's `lines` list, with that order's `order` and `vendor_id` carried along on the row._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

# one row per line item: record_path walks into 'lines', meta copies the parent order fields down
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
print(lines)

assert len(lines) == 4, len(lines)
assert lines['qty'].sum() == 7, lines['qty'].sum()
print(f'rows = {len(lines)}, total qty = {lines["qty"].sum()}')

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07
rows = 4, total qty = 7


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# vendors_json is already flat (one dict per vendor, no nesting), so json_normalize just makes the frame
vendors = pd.json_normalize(vendors_json)
print(vendors, '\n')

# guard: a duplicated vendor_id on the right side would fan out line rows and inflate qty totals
dupes = vendors[vendors['vendor_id'].duplicated(keep=False)]
print('duplicate vendor_ids in vendors:', dupes['vendor_id'].unique().tolist() or 'none')

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True,
                     validate='many_to_one')  # raises if vendors ever has duplicate keys
print(joined, '\n')
print(joined['_merge'].value_counts(), '\n')

unmatched = joined[joined['_merge'] == 'left_only']
print('unmatched vendor_id(s):', unmatched['vendor_id'].unique().tolist())
print('units attached to unmatched vendor(s):', unmatched['qty'].sum())
print('row count unchanged by join:', len(joined) == len(lines))

  vendor_id          name zone
0      V-01  Hoos Burgers    A
1      V-10     Cav Merch    A 

duplicate vendor_ids in vendors: none
           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only 

_merge
both          3
left_only     1
right_only    0
Name: count, dtype: int64 

unmatched vendor_id(s): ['V-07']
units attached to unmatched vendor(s): 3
row count unchanged by join: True


**Q2 answer:** `V-07` is the only vendor_id with no match in the vendor list (`_merge == 'left_only'`). It belongs to order 3: 1 line, **3 units** (Hot Dog x 3). `vendors` has no duplicate `vendor_id`s, so the left join keeps exactly 4 rows. `validate='many_to_one'` would raise if a duplicate ever showed up and started multiplying line rows.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Choice: keep the unmatched vendor as its own row labelled 'UNKNOWN (V-07)' instead of dropping it.
# Why: groupby drops NaN keys by default, which would silently lose 3 of the 7 units. Labelling it
# with its vendor_id keeps the table summing to 7 and tells the reader exactly which ID needs a
# vendor-list fix, rather than lumping it into an anonymous NaN bucket.
vendor_label = joined['name'].fillna('UNKNOWN (' + joined['vendor_id'] + ')')
qty_by_vendor = (joined.assign(vendor_name=vendor_label)
                       .groupby('vendor_name')['qty'].sum()
                       .sort_values(ascending=False))
print(qty_by_vendor)
print('total:', qty_by_vendor.sum())
assert qty_by_vendor.sum() == lines['qty'].sum()

vendor_name
Hoos Burgers      3
UNKNOWN (V-07)    3
Cav Merch         1
Name: qty, dtype: int64
total: 7


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# Zone comes only from the vendor list, so V-07's rows have zone = NaN.
# groupby's default dropna=True drops them. That's fine for "which known zone sold most",
# but I account for the gap explicitly instead of letting it vanish.
qty_by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)
print(qty_by_zone, '\n')

zone_total = qty_by_zone.sum()
no_zone = joined.loc[joined['zone'].isna(), 'qty'].sum()
print(f'zone with most items: {qty_by_zone.idxmax()} ({qty_by_zone.max()} units)')
print(f'zone table sums to {zone_total}, not {lines["qty"].sum()}: '
      f'{no_zone} units belong to unmatched vendor(s) '
      f'{joined.loc[joined["zone"].isna(), "vendor_id"].unique().tolist()}, which have no zone')
assert zone_total + no_zone == lines['qty'].sum()

# same view with the gap visible as its own row
print()
print(joined.groupby('zone', dropna=False)['qty'].sum())

zone
A    4
Name: qty, dtype: int64 

zone with most items: A (4 units)
zone table sums to 4, not 7: 3 units belong to unmatched vendor(s) ['V-07'], which have no zone

zone
A      4
NaN    3
Name: qty, dtype: int64


**Q4 answer:** Zone **A** sold the most, with **4 units** (Hoos Burgers 3 + Cav Merch 1). It is also the only zone the vendor list knows about. My zone table sums to **4, not 7**. The missing **3 units** are the Hot Dogs from order 3, vendor `V-07`. That vendor isn't in the vendor list, so it has no zone, and `groupby('zone')` drops the NaN key. With `dropna=False` the 3 units show up as their own NaN-zone row.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# json_normalize(record_path='lines') raises KeyError on order 5, so split first.
# An order with no 'lines' has no line items, so it doesn't belong in a line-item table.
# I keep it in a separate frame so it's tracked, not silently dropped.
has_lines = [o for o in ragged_json if o.get('lines')]
no_lines  = pd.json_normalize([o for o in ragged_json if not o.get('lines')])

ragged_lines = pd.json_normalize(has_lines, record_path='lines', meta=['order', 'vendor_id'])
# missing qty -> NaN after normalize; cast to nullable Int64 so it shows as <NA>, not 0 (and not 2.0)
ragged_lines['qty'] = ragged_lines['qty'].astype('Int64')
print(ragged_lines, '\n')
print(ragged_lines.dtypes, '\n')

print('lines with missing qty:', ragged_lines['qty'].isna().sum(),
      '| lines with qty == 0:', (ragged_lines['qty'] == 0).sum())
print('known qty total (missing skipped, not zeroed):', ragged_lines['qty'].sum())
print('\norders with no lines (tracked separately):')
print(no_lines)

    item   qty order vendor_id
0   Soda     2     4      V-01
1  Fries  <NA>     6      V-01 

item         object
qty           Int64
order        object
vendor_id    object
dtype: object 

lines with missing qty: 1 | lines with qty == 0: 0
known qty total (missing skipped, not zeroed): 2

orders with no lines (tracked separately):
   order vendor_id
0      5      V-10


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** The thing we actually count is items, and items live on the lines, not on the orders. With one row per line item, the 7 units are just `qty.sum()`. Questions like "how many units did each vendor or zone sell" or "what's our best-selling item" each take one `groupby`. If each row were an order, order 1 would be a single row holding a list (`Cheeseburger x2, Soda x1`). Any per-item question would mean unpacking those lists first, and counting rows would give you 3 orders, not 7 units. The line-level shape also made the data-quality problem obvious: after the join, 3 of the 7 units (the Hot Dogs on order 3) had no vendor. You can still answer order-level questions by grouping on `order`. You can't get line-level answers out of an order-level table without flattening it again.

**b)** A quantity of 0 is a fact: the line was recorded and nobody bought anything. A missing quantity means we don't know. The Fries line on order 6 could have been 1 or 10. If you `fillna(0)`, you turn "unknown" into a confident "nothing was sold", and the line effectively disappears downstream. Totals look complete: batch 2 would report 2 units when the real answer is "2, plus an unknown number of Fries." The average quantity per line drops because a fake 0 gets counted. Fries looks like a product that doesn't sell. And nobody goes back to the vendor or the source API to fix the record. Keeping the value as `<NA>` in an `Int64` column keeps the gap visible. `isna()` finds it, `sum()` skips it instead of treating it as 0, and the report can say "2 known units + 1 line with unknown qty." Order 5 is the same kind of problem one level up. It has no lines at all, so I kept it in its own table instead of letting it quietly drop out.